# Notebook 03 — Cross-Domain Fairness Transfer (BRSET to mBRSET)
**BECITHCON 2026 · Fairness under domain shift · Experiment 3 of 4**

Notebooks 01 and 02 established that the clinic-camera model has subgroup disparities, that
image quality is the worst axis, and that cheap post-hoc fixes are partial and fragile. This
notebook asks the central question of the paper: **do those disparities survive when the
model is deployed zero-shot on the portable camera (mBRSET)?**

We evaluate on the two labels the datasets share, **diabetic retinopathy** (mBRSET
`final_icdr` >= 1) and **macular edema** (mBRSET `final_edema`), across the axes both
datasets carry, **sex, image quality, and age**. Camera is not an axis here; it *is* the
domain shift.

**Deployment is zero-shot:** the operating threshold is learned on BRSET validation and
applied unchanged to mBRSET. We lead on the **AUC gap** (threshold-free, asks whether ranking
fairness transfers) and report **sensitivity/FPR gaps** as the operating-point view.

**You fill in:** the mBRSET image directory, and the mBRSET sex mapping (CONFIRM from the
mBRSET data dictionary — BRSET is confirmed 1=male, 2=female).


In [ ]:
# ============================== CONFIG ==============================
BRSET_LABELS  = "labels_brset.csv"
SPLIT_FILE    = "split.csv"
BRSET_PREDS   = "fairness_outputs/brset_val_test_preds.csv"   # from notebook 01

MBRSET_LABELS    = "labels_mbrset.csv"
MBRSET_IMAGE_DIR = "/path/to/mbrset/images"                   # <-- fill in
CHECKPOINT       = "/path/to/weighted_bce.pt"                 # same model as notebook 01

# Sex harmonisation to a common male/female label across datasets.
BRSET_SEX_MAP  = {1: 'male', 2: 'female'}        # CONFIRMED from BRSET paper counts
MBRSET_SEX_MAP = {0: 'female', 1: 'male'}        # <-- PLACEHOLDER. CONFIRM from mBRSET dict.
MBRSET_SEX_CONFIRMED = False                      # set True once you have verified the line

OUTPUT_DIR   = "fairness_outputs"
DEVICE       = "cuda"
SEED         = 42
IMG_SIZE     = 224
BATCH_SIZE   = 64
NUM_WORKERS  = 4
TARGET_SENSITIVITY = 0.85
N_BOOTSTRAP        = 1000
ECE_BINS           = 10


In [ ]:
import os, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore"); np.random.seed(SEED)
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not MBRSET_SEX_CONFIRMED:
    print("*** mBRSET sex mapping is a PLACEHOLDER. Confirm it in the data dictionary and")
    print("*** set MBRSET_SEX_CONFIRMED=True. Results are valid except for which sex is named.")

# The two shared labels. Model output column names come from notebook 01's LABELS order.
SHARED = ['diabetic_retinopathy', 'macular_edema']
AXES   = {'sex': 'sex_group', 'quality': 'quality_group', 'age': 'age_group'}


## Section 1 — BRSET side: deployed thresholds and in-domain DR/ME fairness

We reload the predictions notebook 01 saved, fix the deployed operating point on BRSET
validation, and recompute the in-domain DR and macular-edema subgroup fairness so we have a
like-for-like baseline to compare against mBRSET.


In [ ]:
def safe_auc(y,p):
    y=np.asarray(y); p=np.asarray(p)
    return roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan
def ece(y,p,n_bins=ECE_BINS):
    y=np.asarray(y); p=np.asarray(p)
    if len(y)==0: return np.nan
    bins=np.linspace(0,1,n_bins+1); e=0.0
    for lo,hi in zip(bins[:-1],bins[1:]):
        m=(p>lo)&(p<=hi)
        if m.sum()==0: continue
        e+=m.mean()*abs(y[m].mean()-p[m].mean())
    return e
def sens_fpr(y,p,thr):
    y=np.asarray(y); pred=(np.asarray(p)>=thr).astype(int)
    tp=((pred==1)&(y==1)).sum(); fn=((pred==0)&(y==1)).sum()
    fp=((pred==1)&(y==0)).sum(); tn=((pred==0)&(y==0)).sum()
    return (tp/(tp+fn) if tp+fn else np.nan, fp/(fp+tn) if fp+tn else np.nan)
def thr_for_sens(y,p,target):
    y=np.asarray(y); p=np.asarray(p)
    if y.sum()==0: return 0.5
    chosen=np.unique(p).min()
    for t in np.unique(p)[::-1]:
        s,_=sens_fpr(y,p,t)
        if s>=target: chosen=t; break
    return float(chosen)

bd = pd.read_csv(BRSET_LABELS).merge(pd.read_csv(SPLIT_FILE)[['patient_id','split']],
                                     on='patient_id', how='left')
bd['sex_group']     = bd['patient_sex'].map(BRSET_SEX_MAP)
bd['quality_group'] = bd['quality']
def age_band(a):
    if pd.isna(a): return np.nan
    return '<40' if a<40 else ('40-59' if a<60 else '60+')
bd['age_group'] = bd['patient_age'].apply(age_band)
bd = bd.merge(pd.read_csv(BRSET_PREDS), on='image_id', how='inner')
bval, btest = bd[bd.split=='val'], bd[bd.split=='test']

DEPLOYED_THR = {l: thr_for_sens(bval[l], bval['prob_'+l], TARGET_SENSITIVITY) for l in SHARED}
print("Deployed thresholds (BRSET val):", {k: round(v,3) for k,v in DEPLOYED_THR.items()})

def subgroup_metrics(frame, label, pid_col):
    rows=[]
    for axis,col in AXES.items():
        for g,gdf in frame.dropna(subset=[col]).groupby(col):
            y=gdf[label].values; p=gdf['prob_'+label].values
            s,f=sens_fpr(y,p,DEPLOYED_THR[label])
            rows.append(dict(axis=axis,group=g,label=label,n=len(gdf),n_pos=int(y.sum()),
                             auc=safe_auc(y,p),sensitivity=s,fpr=f,ece=ece(y,p)))
    return pd.DataFrame(rows)

brset_metrics = pd.concat([subgroup_metrics(btest,l,'patient_id') for l in SHARED])
print("\nBRSET in-domain DR/ME subgroup metrics computed.")


## Section 2 — mBRSET side: derive shared labels and subgroups

In [ ]:
md = pd.read_csv(MBRSET_LABELS)
md['diabetic_retinopathy'] = (pd.to_numeric(md['final_icdr'], errors='coerce') >= 1).astype('float')
md.loc[pd.to_numeric(md['final_icdr'], errors='coerce').isna(), 'diabetic_retinopathy'] = np.nan
md['macular_edema'] = md['final_edema'].map({'yes':1.0, 'no':0.0})

md['sex_group']     = md['sex'].map(MBRSET_SEX_MAP)
md['quality_group'] = md['final_quality'].map({'yes':'Adequate', 'no':'Inadequate'})
md['age_num']       = pd.to_numeric(md['age'], errors='coerce')
md['age_group']     = md['age_num'].apply(age_band)
md['image_id']      = md['file']

print("mBRSET shared-label prevalence:")
for l in SHARED:
    s=md[l]; print(f"  {l}: pos={int(s.sum(skipna=True))}  labelled={int(s.notna().sum())}")
print("quality:", md['quality_group'].value_counts(dropna=False).to_dict())
print("sex:", md['sex_group'].value_counts(dropna=False).to_dict())


## Section 3 — Run the BRSET model on mBRSET (zero-shot inference)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

LABELS_ORDER = ['diabetic_retinopathy','macular_edema','scar','nevus','amd',
                'vascular_occlusion','hypertensive_retinopathy','drusens','hemorrhage',
                'myopic_fundus','increased_cup_disc']

class BRSETClassifier(nn.Module):
    def __init__(self, n_labels):
        super().__init__()
        try: self.backbone = models.efficientnet_b0(weights=None)
        except TypeError: self.backbone = models.efficientnet_b0(pretrained=False)
        in_f = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.head = nn.Linear(in_f, n_labels)
    def forward(self, x): return self.head(self.backbone(x))

_tf = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor(),
                          transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

class MBDataset(Dataset):
    def __init__(self, frame, d): self.frame=frame.reset_index(drop=True); self.d=d
    def __len__(self): return len(self.frame)
    def _find(self, f):
        for ext in ('', '.jpg', '.jpeg', '.png'):
            p=os.path.join(self.d, str(f)+ext)
            if os.path.exists(p): return p
        raise FileNotFoundError(f)
    def __getitem__(self,i):
        f=self.frame.loc[i,'file']; return _tf(Image.open(self._find(f)).convert('RGB')), str(f)

@torch.no_grad()
def infer(frame):
    loader=DataLoader(MBDataset(frame, MBRSET_IMAGE_DIR), batch_size=BATCH_SIZE,
                      shuffle=False, num_workers=NUM_WORKERS)
    ids,probs=[],[]
    for x,b in loader:
        probs.append(torch.sigmoid(model(x.to(dev))).cpu().numpy()); ids.extend(b)
    out=pd.DataFrame(np.concatenate(probs,0), columns=['prob_'+l for l in LABELS_ORDER])
    out['file']=ids; return out

dev = DEVICE if (DEVICE=='cpu' or torch.cuda.is_available()) else 'cpu'
model = BRSETClassifier(len(LABELS_ORDER)).to(dev)
ck = torch.load(CHECKPOINT, map_location=dev); st = ck.get('model_state_dict', ck) if isinstance(ck,dict) else ck
model.load_state_dict(st); model.eval()
mb_pred = infer(md)
mb_pred.to_csv(os.path.join(OUTPUT_DIR,'mbrset_preds.csv'), index=False)
mdata = md.merge(mb_pred[['file','prob_diabetic_retinopathy','prob_macular_edema']], on='file', how='inner')
print("mBRSET inference done:", mdata.shape)


## Section 4 — mBRSET subgroup fairness (out-of-domain)

In [ ]:
def subgroup_metrics_mb(frame, label):
    rows=[]
    sub=frame[frame[label].notna()]
    for axis,col in AXES.items():
        for g,gdf in sub.dropna(subset=[col]).groupby(col):
            y=gdf[label].values; p=gdf['prob_'+label].values
            s,f=sens_fpr(y,p,DEPLOYED_THR[label])
            rows.append(dict(axis=axis,group=g,label=label,n=len(gdf),n_pos=int(y.sum()),
                             auc=safe_auc(y,p),sensitivity=s,fpr=f,ece=ece(y,p)))
    return pd.DataFrame(rows)

mbrset_metrics = pd.concat([subgroup_metrics_mb(mdata,l) for l in SHARED])
print("mBRSET out-of-domain subgroup metrics:")
print(mbrset_metrics.round(3).to_string(index=False))


## Section 5 — The headline: does the gap survive the shift?

For each shared label and axis we put the in-domain (BRSET) gap beside the out-of-domain
(mBRSET) gap, for AUC (ranking fairness) and sensitivity (operating-point fairness).


In [ ]:
def gaps_from(metrics, label, axis, metric):
    v=metrics[(metrics.label==label)&(metrics.axis==axis)][metric].dropna()
    return (v.max()-v.min()) if len(v)>=2 else np.nan

rows=[]
for l in SHARED:
    for axis in AXES:
        for metric in ['auc','sensitivity','fpr','ece']:
            rows.append(dict(label=l, axis=axis, metric=metric,
                in_domain=gaps_from(brset_metrics,l,axis,metric),
                out_domain=gaps_from(mbrset_metrics,l,axis,metric)))
compare=pd.DataFrame(rows)
compare['change']=compare['out_domain']-compare['in_domain']
print("Fairness gap: in-domain (BRSET) vs out-of-domain (mBRSET)")
print(compare.round(3).to_string(index=False))


## Section 6 — Overall transfer effect (the aggregate domain shift)

Separate from fairness, how much does overall performance move under the shift? This is the
context for the gap changes above.


In [ ]:
rows=[]
for l in SHARED:
    yb=btest[l].values; pb=btest['prob_'+l].values
    sub=mdata[mdata[l].notna()]; ym=sub[l].values; pm=sub['prob_'+l].values
    sb,fb=sens_fpr(yb,pb,DEPLOYED_THR[l]); sm,fm=sens_fpr(ym,pm,DEPLOYED_THR[l])
    rows.append(dict(label=l, auc_brset=safe_auc(yb,pb), auc_mbrset=safe_auc(ym,pm),
                     sens_brset=sb, sens_mbrset=sm, fpr_brset=fb, fpr_mbrset=fm,
                     ece_brset=ece(yb,pb), ece_mbrset=ece(ym,pm)))
overall=pd.DataFrame(rows)
print("Overall performance, BRSET test vs mBRSET zero-shot:")
print(overall.round(3).to_string(index=False))


## Section 7 — Bootstrap CIs on the out-of-domain gaps (patient-level)

In [ ]:
def boot_gap_mb(frame, label, axis, metric, B=N_BOOTSTRAP, seed=SEED):
    rng=np.random.default_rng(seed); col=AXES[axis]
    sub=frame[frame[label].notna()]
    by={p:idx.values for p,idx in sub.groupby('patient').groups.items()}
    pts=sub['patient'].unique(); vals=[]
    for _ in range(B):
        samp=rng.choice(pts,len(pts),replace=True)
        bf=sub.loc[np.concatenate([by[p] for p in samp])]
        per=[]
        for g,gdf in bf.dropna(subset=[col]).groupby(col):
            y=gdf[label].values; p=gdf['prob_'+label].values
            if metric=='auc': v=safe_auc(y,p)
            elif metric=='ece': v=ece(y,p)
            else:
                s,f=sens_fpr(y,p,DEPLOYED_THR[label]); v=s if metric=='sensitivity' else f
            if not np.isnan(v): per.append(v)
        if len(per)>=2: vals.append(max(per)-min(per))
    return (np.mean(vals),np.percentile(vals,2.5),np.percentile(vals,97.5)) if vals else (np.nan,)*3

ci=[]
for l in SHARED:
    for axis in AXES:
        for metric in ['auc','sensitivity']:
            m,lo,hi=boot_gap_mb(mdata,l,axis,metric)
            ci.append(dict(label=l,axis=axis,metric=metric,gap=m,ci_low=lo,ci_high=hi,
                           excludes_zero=(lo>0)))
ci=pd.DataFrame(ci)
print("Out-of-domain gaps whose 95% CI excludes zero:")
print(ci[ci.excludes_zero].round(3).to_string(index=False))


## Section 8 — Figure and export

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,metric in zip(axes,['auc','sensitivity']):
    sub=compare[(compare.metric==metric)]
    lab=[f"{r.label[:8]}/{r.axis}" for r in sub.itertuples()]
    x=np.arange(len(sub)); w=0.38
    ax.bar(x-w/2, sub['in_domain'], w, label='BRSET (in-domain)')
    ax.bar(x+w/2, sub['out_domain'], w, label='mBRSET (out-of-domain)')
    ax.set_xticks(x); ax.set_xticklabels(lab, rotation=45, ha='right', fontsize=8)
    ax.set_title(f'{metric} gap: in- vs out-of-domain'); ax.set_ylabel(f'{metric} gap')
axes[0].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'figE_crossdomain_gaps.png'),dpi=200); plt.close()

brset_metrics.to_csv(os.path.join(OUTPUT_DIR,'brset_DRME_metrics.csv'),index=False)
mbrset_metrics.to_csv(os.path.join(OUTPUT_DIR,'mbrset_DRME_metrics.csv'),index=False)
compare.to_csv(os.path.join(OUTPUT_DIR,'crossdomain_gap_compare.csv'),index=False)
overall.to_csv(os.path.join(OUTPUT_DIR,'crossdomain_overall.csv'),index=False)
ci.to_csv(os.path.join(OUTPUT_DIR,'crossdomain_mb_ci.csv'),index=False)
print("Saved figE_crossdomain_gaps.png and five CSVs to", OUTPUT_DIR)


## Section 9 — Reading the cross-domain results (for the paper)

- **The headline is Section 5's `change` column.** A positive change means the disparity
  *widened* under the portable-camera shift, which is the paper's hypothesis. Lead on the
  **AUC** rows (ranking fairness) and support with **sensitivity** (operating-point fairness).
- **Read it against Section 6.** If overall performance also dropped, frame it as "the shift
  hurts everyone, but not equally." If overall held but a gap widened, that is an even
  sharper equity finding.
- **Quality is the axis to watch.** It was the worst in-domain and a portable camera produces
  more inadequate images, so a widening quality gap is the cleanest through-line of the paper.
- **Tie back to notebooks 01-02.** The per-group cells get smaller out-of-domain, so the same
  small-sample fragility that broke AMD mitigation predicts that any per-group fix will be
  even shakier here. That motivates notebook 04, which tests whether adaptation that restores
  overall accuracy preserves or damages fairness.
- **Naming:** once you confirm the mBRSET sex code, the sex rows can name male/female; until
  then describe sex effects by the harmonised label and flag it.
